In [1]:
!pip install num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 6.8 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=b4e9687dde7da7a77340a7dfc0781798227d6099cbf495e654869a4b4b8c9232
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [2]:
from num2words import num2words
import random

def generate_data(n_samples=10000):
    inputs = []
    outputs = []

    for _ in range(n_samples):
        num = random.randint(0, 9999)  # control difficulty here
        inputs.append(str(num))
        outputs.append(num2words(num))

    return inputs, outputs

questions, answers = generate_data(10000)

In [3]:
random.randint(0, 999)     # EASY
random.randint(0, 9999)    # MEDIUM
random.randint(0, 99999)   # HARD
# Note: The model training error concerning vocabulary size should be addressed in the tokenizer and model definition cells, not here.

16523

In [4]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[-,]', ' ', text)  # remove hyphens/commas
    text = re.sub(r'\s+', ' ', text).strip()
    return text

answers = [clean_text(a) for a in answers]

In [5]:
answers = ["<start> " + a + " <end>" for a in answers]

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Initialize tokenizer for character-level tokenization
tokenizer = Tokenizer(char_level=True)

# Fit tokenizer on both questions and answers to build the character vocabulary
all_text = questions + answers
tokenizer.fit_on_texts(all_text)

# Determine maximum sequence length for character-level tokenization
# This is crucial as numbers in words can be long (e.g., 'one hundred ninety-nine')
# Get all sequences to find the max length
temp_sequences = tokenizer.texts_to_sequences(all_text)
max_char_seq_len = max(len(seq) for seq in temp_sequences)

# Add a small buffer to max_char_seq_len for safety, e.g., +5
maxlen = max_char_seq_len + 5

print(f"Calculated max character sequence length: {max_char_seq_len}")
print(f"Using padding maxlen: {maxlen}")


# Convert text to sequences using the fitted character-level tokenizer
q_seq = tokenizer.texts_to_sequences(questions)
a_seq = tokenizer.texts_to_sequences(answers)

# Pad sequences to the determined fixed length
q_seq = pad_sequences(q_seq, maxlen=maxlen, padding='post')
a_seq = pad_sequences(a_seq, maxlen=maxlen, padding='post')

Calculated max character sequence length: 60
Using padding maxlen: 65


In [7]:
# tokenizer = Tokenizer(char_level=True) # Redundant after modification in P8odsj7Pd9oK

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    q_seq, a_seq, test_size=0.1
)

In [9]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

vocab_size = len(tokenizer.word_index) + 1

# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(vocab_size, 256)(encoder_inputs)
encoder_lstm = LSTM(256, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(vocab_size, 256)(decoder_inputs)
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

decoder_dense = Dense(vocab_size, activation='softmax')
output = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

In [10]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

vocab_size = len(tokenizer.word_index) + 1

# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(vocab_size, 256)(encoder_inputs)
encoder_lstm = LSTM(256, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(vocab_size, 256)(decoder_inputs)
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

decoder_dense = Dense(vocab_size, activation='softmax')
output = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=["accuracy"])

In [11]:
from keras.callbacks import EarlyStopping
model.fit(
    [X_train, y_train[:, :-1]],
    y_train[:, 1:],
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[EarlyStopping(patience=3)]
)

Epoch 1/50
254/254 ━━━━━━━━━━━━━━━━━━━━ 156s 598ms/step - accuracy: 0.8260 - loss: 0.5793 - val_accuracy: 0.9306 - val_loss: 0.1582
Epoch 2/50
254/254 ━━━━━━━━━━━━━━━━━━━━ 153s 604ms/step - accuracy: 0.9314 - loss: 0.1524 - val_accuracy: 0.9316 - val_loss: 0.1507
Epoch 3/50
254/254 ━━━━━━━━━━━━━━━━━━━━ 199s 592ms/step - accuracy: 0.9315 - loss: 0.1482 - val_accuracy: 0.9325 - val_loss: 0.1473
Epoch 4/50
254/254 ━━━━━━━━━━━━━━━━━━━━ 152s 600ms/step - accuracy: 0.9321 - loss: 0.1471 - val_accuracy: 0.9327 - val_loss: 0.1459
Epoch 5/50
254/254 ━━━━━━━━━━━━━━━━━━━━ 214s 648ms/step - accuracy: 0.9323 - loss: 0.1467 - val_accuracy: 0.9362 - val_loss: 0.1445
Epoch 6/50
254/254 ━━━━━━━━━━━━━━━━━━━━ 197s 630ms/step - accuracy: 0.9386 - loss: 0.1372 - val_accuracy: 0.9399 - val_loss: 0.1311
Epoch 7/50
254/254 ━━━━━━━━━━━━━━━━━━━━ 204s 637ms/step - accuracy: 0.9417 - loss: 0.1277 - val_accuracy: 0.9430 - val_loss: 0.1257
Epoch 8/50
254/254 ━━━━━━━━━━━━━━━━━━━━ 159s 625ms/step - accuracy: 0.9433 -

In [12]:
loss,accuracy=model.evaluate([X_test,y_test[:,:-1]],y_test[:,1:])

32/32 ━━━━━━━━━━━━━━━━━━━━ 7s 204ms/step - accuracy: 0.9507 - loss: 0.1031


In [13]:
loss

0.10307201743125916

In [14]:
accuracy

0.9507031440734863